## Tools
models can request to call tools that perform tasks such as fetching data from database, searching the web, or running code. Tools are pairings of:
    1. A schema, including the name of the tool, a description, and/or argument definitions (often a JSON shcema)
    2. A function or coroutine to execute.

In [1]:
import os
from langchain_groq import ChatGroq

os.environ["GROQ_API_KEY"]= os.getenv("GROQ_API_KEY")

model=ChatGroq(model="llama-3.3-70b-versatile")

In [2]:
from langchain.tools import tool

@tool #decorator for tool 
def GetTemp(Location:str)->str:
    """helps in getting the temprature at a given location"""
    return f"Temp in {Location} is 100 degree celcius."

model_withTool=model.bind_tools([GetTemp])



In [3]:
response= model_withTool.invoke("what is the temprature in Jamshedpur?")
print(response)
for tool_call in response.tool_calls:
    #view tool calls made by the model
    print(f"Tool: {tool_call['name']}")
    print(f"Args: {tool_call['args']}")

content='' additional_kwargs={'tool_calls': [{'id': '2fdbwfg30', 'function': {'arguments': '{"Location":"Jamshedpur"}', 'name': 'GetTemp'}, 'type': 'function'}]} response_metadata={'token_usage': {'completion_tokens': 17, 'prompt_tokens': 235, 'total_tokens': 252, 'completion_time': 0.070727197, 'completion_tokens_details': None, 'prompt_time': 0.011557067, 'prompt_tokens_details': None, 'queue_time': 0.162052595, 'total_time': 0.082284264}, 'model_name': 'llama-3.3-70b-versatile', 'system_fingerprint': 'fp_45180df409', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'} id='lc_run--019fd689-d82a-7732-a624-244e6d537887-0' tool_calls=[{'name': 'GetTemp', 'args': {'Location': 'Jamshedpur'}, 'id': '2fdbwfg30', 'type': 'tool_call'}] invalid_tool_calls=[] usage_metadata={'input_tokens': 235, 'output_tokens': 17, 'total_tokens': 252}
Tool: GetTemp
Args: {'Location': 'Jamshedpur'}


or

In [5]:
from langchain.agents import create_agent

def GetTempp(Location:str)->str:
    """helps in getting the temprature at a given location"""
    return f"Temp in {Location} is 100 degree celcius."

agent=create_agent(
    model="groq:llama-3.3-70b-versatile",
    tools=[GetTempp],
    system_prompt="helpful assistant"
)

In [ ]:
response = agent.invoke(
    {
        "messages": [
            {
                "role": "user",
                "content": "temperature in Jamshedpur?"
            }
        ]
    }
)

print(response)
#glt hai-  print(response.content)
response["messages"][-1].content


### Tool execution Loops

In [ ]:
#step 1: model generates tool calls
messages= [{"role":"user", "content": "what's the temprature in India?"}]
ai_msg=model_withTool.invoke(messages)
messages.append(ai_msg)

#step 2: Execute tools and collect results
for tool_call in ai_msg.tool_calls:
    #execure the rool with the generated arguments
    tool_result=GetTemp.invoke(tool_call)
    messages.append(tool_result)

#step 3: Pass results back to model for final response
final_response= model_withTool.invoke(messages)
print(final_response.text)
